# Actor annotation results: scores + actor counts per language

Run this **after** finishing the manual checks. It reads the `comparison` sheet of
each file and writes **two APA-ready Excel tables**:

* **`actor_results_model_scores.xlsx`** — precision, recall, F1, agreement, n articles
* **`actor_results_nr_actors.xlsx`** — actors coded by human vs gpt + matched / missed / extra

**English is split into two files** (`_UK` and `_IE`); they are merged here into one
`English` row. Dutch (NL+BE) and Turkish are single files.

Definitions (one row per actor mention):
* **Matched (TP)** coded by both · **Missed by GPT (FN)** `missedbygpt=1` · **Coded extra by GPT (FP)** `coded extra=1`
* **precision** = matched / all gpt actors · **recall** = matched / all human actors
* **F1** = 2·P·R/(P+R) · **agreement** = matched / (matched + missed + extra)

In [ ]:
import pandas as pd

# each language -> one or more comparison files (English merges UK + Ireland)
FILES = {
    "English": ["human_gpt_comparison_actors_english_UK.xlsx",
                "human_gpt_comparison_actors_english_IE.xlsx"],
    "Dutch":   ["human_gpt_comparison_actors_dutch.xlsx"],
    "Turkish": ["human_gpt_comparison_actors_turkish.xlsx"],
}

In [ ]:
def counts_for(paths):
    # read + stack the comparison sheet(s) for this language
    df = pd.concat([pd.read_excel(p, sheet_name="comparison") for p in paths],
                   ignore_index=True)
    df["missedbygpt"] = pd.to_numeric(df["missedbygpt"], errors="coerce")
    df["coded extra"] = pd.to_numeric(df["coded extra"], errors="coerce")

    human = df[df["coder"] == "human"]
    gpt   = df[df["coder"] == "gpt_oss"]

    matched_h = int((human["missedbygpt"] == 0).sum())   # for recall
    missed    = int((human["missedbygpt"] == 1).sum())   # FN
    matched_g = int((gpt["coded extra"] == 0).sum())     # for precision
    extra     = int((gpt["coded extra"] == 1).sum())     # FP

    if matched_h != matched_g:
        print(f"  CHECK ({', '.join(paths)}): matched differs human={matched_h} "
              f"gpt={matched_g} -> a flag was edited on only one side.")

    return {
        "n_articles":   df["article_id"].nunique(),
        "human_actors": len(human),
        "gpt_actors":   len(gpt),
        "matched":      matched_h,
        "missed":       missed,
        "extra":        extra,
        "matched_gpt":  matched_g,
    }


raw = pd.DataFrame({lang: counts_for(paths) for lang, paths in FILES.items()}).T
raw

In [ ]:
# ---- Table 1: model scores (APA) ----
precision = raw["matched_gpt"] / raw["gpt_actors"]
recall    = raw["matched"]     / raw["human_actors"]
f1        = 2 * precision * recall / (precision + recall)
agreement = raw["matched"]     / (raw["matched"] + raw["missed"] + raw["extra"])

scores = pd.DataFrame({
    "Articles":  raw["n_articles"].astype(int),
    "Precision": precision.round(2),
    "Recall":    recall.round(2),
    "F1":        f1.round(2),
    "Agreement": agreement.round(2),
})
scores.index.name = "Language"
scores

In [ ]:
# ---- Table 2: number of actors (APA) ----
nr_actors = pd.DataFrame({
    "Articles":           raw["n_articles"].astype(int),
    "Human actors":       raw["human_actors"].astype(int),
    "GPT actors":         raw["gpt_actors"].astype(int),
    "Matched":            raw["matched"].astype(int),
    "Missed by GPT":      raw["missed"].astype(int),
    "Coded extra by GPT": raw["extra"].astype(int),
})
nr_actors.index.name = "Language"
nr_actors

In [ ]:
# ---- write the two APA tables ----
scores.to_excel("actor_results_model_scores.xlsx")
nr_actors.to_excel("actor_results_nr_actors.xlsx")
print("saved -> actor_results_model_scores.xlsx")
print("saved -> actor_results_nr_actors.xlsx")